In [ ]:
taxon_id_soup_dict = {}
with open('taxonid_list.txt') as taxon_id_list:
    for taxon_id in taxon_id_list:
        url = f"https://meta-virome.org/MetaData/{taxon_id}"
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
        }

        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "lxml")
        taxon_id_soup_dict[taxon_id] = soup




In [ ]:
import csv
import time
import requests
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

FIELDS = [
    "Study Name",
    "Ncbi Phylum",
    "Ncbi Class",
    "Ncbi Order",
    "Ncbi Family",
    "Ncbi Genus",
    "Ncbi Species",
    "Host Name",
    "Ncbi Genbank Id",
    "Geographic Location",
    "Isolation Country",
    "Sample Collection Date",
    "Add Date",
    "Public Release Date",
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)


def parse_metadata(soup: BeautifulSoup) -> dict:
    """
    Walk every row of every meta-data table and build a {label: value} dict.
    Only picks up rows that have exactly two <td> cells.
    """
    data = {}
    for table in soup.select("table.meta-data-table"):
        for tr in table.find_all("tr"):
            tds = tr.find_all("td")
            if len(tds) != 2:
                continue
            key = tds[0].get_text(strip=True)
            val = tds[1].get_text(" ", strip=True)
            if key and key not in data:
                data[key] = val
    return data


def scrape_one(taxon_id: str, retries: int = 2) -> dict:
    url = f"https://meta-virome.org/MetaData/{taxon_id}"
    last_err = None
    for attempt in range(retries + 1):
        try:
            r = SESSION.get(url, timeout=20)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "lxml")
            parsed = parse_metadata(soup)
            row = {"Taxon ID": taxon_id}
            for f in FIELDS:
                row[f] = parsed.get(f, "")
            return row
        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1)) 
    raise last_err


def run_pass(taxon_ids, max_workers, label):
    """
    Scrape a list of taxon IDs in parallel.
    Returns (results_dict, failed_ids_list).
    """
    results = {}
    failed = []
    total = len(taxon_ids)

    if total == 0:
        return results, failed

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        future_to_id = {pool.submit(scrape_one, tid): tid for tid in taxon_ids}
        done = 0
        for fut in as_completed(future_to_id):
            tid = future_to_id[fut]
            done += 1
            try:
                results[tid] = fut.result()
                print(f"[{label} {done}/{total}] OK   {tid}", flush=True)
            except Exception as e:
                results[tid] = {"Taxon ID": tid, **{f: "" for f in FIELDS}}
                failed.append(tid)
                print(f"[{label} {done}/{total}] FAIL {tid}: {e}", flush=True)

    return results, failed


def main():
    with open("taxonid_list.txt") as f:
        taxon_ids = [line.strip() for line in f if line.strip()]

    MAX_WORKERS = 4         
    RETRY_WORKERS = 2       
    RETRY_DELAY = 5          
    MAX_RETRY_PASSES = 3     

    all_results = {}

    print(f"=== Pass 1: scraping {len(taxon_ids)} IDs with {MAX_WORKERS} workers ===")
    results, failed = run_pass(taxon_ids, MAX_WORKERS, "pass1")
    all_results.update(results)

    for pass_num in range(2, MAX_RETRY_PASSES + 2):
        if not failed:
            break
        print(f"\n=== Pass {pass_num}: retrying {len(failed)} failed IDs "
              f"after {RETRY_DELAY}s delay ===")
        time.sleep(RETRY_DELAY)
        results, failed = run_pass(failed, RETRY_WORKERS, f"pass{pass_num}")
        all_results.update(results)
        RETRY_DELAY *= 2 

    if failed:
        with open("failed_taxonids.txt", "w") as f:
            for tid in failed:
                f.write(f"{tid}\n")
        print(f"\n{len(failed)} IDs still failed after all retries. "
              f"Written to failed_taxonids.txt")
    else:
        print("\nAll IDs scraped successfully.")
        import os
        if os.path.exists("failed_taxonids.txt"):
            os.remove("failed_taxonids.txt")

    with open("metaVR_astroviridae_parsed_metadata.csv", "w", newline="", encoding="utf-8") as fout:
        writer = csv.DictWriter(fout, fieldnames=["Taxon ID"] + FIELDS)
        writer.writeheader()
        for tid in taxon_ids:
            writer.writerow(all_results[tid])


if __name__ == "__main__":
    main()

=== Pass 1: scraping 1457 IDs with 4 workers ===
[pass1 1/1457] OK   638276196
[pass1 2/1457] OK   638276540
[pass1 3/1457] OK   638275778
